In [1]:
import os
import json
import glob
import argparse
import pandas as pd
from datetime import datetime

def load_run_data(results_dir):
    """
    Walks through the results directory to find all trajectory.json files
    and aggregates them into a DataFrame.
    """
    data = []
    # Look for trajectory.json files recursively
    search_path = os.path.join(results_dir, "**", "trajectory.json")
    files = glob.glob(search_path, recursive=True)
    
    print(f"Scanning {results_dir}...")
    print(f"Found {len(files)} trajectory files.")

    for filepath in files:
        try:
            with open(filepath, 'r') as f:
                traj = json.load(f)
                
                # Extract key metrics based on your updated run_eval.py structure
                data.append({
                    "Task ID": traj.get("task_id", "Unknown"),
                    "Domain": traj.get("domain", "Unknown"),
                    "Status": traj.get("status", "Unknown"),
                    "Success": 1 if traj.get("status") == "SUCCESS" else 0,
                    "Steps": traj.get("steps_taken", 0),
                    "Failure Reason": traj.get("failure_reason", "None"),
                    "Instruction": traj.get("instruction", "")
                })
        except Exception as e:
            print(f"Error reading {filepath}: {e}")

    return pd.DataFrame(data)

def generate_summary_text(df):
    if df.empty:
        return "No results found. Please check your directory path."

    # 1. Compute Simple Success Rate
    total_tasks = len(df)
    success_count = df['Success'].sum()
    success_rate = (success_count / total_tasks) * 100 if total_tasks > 0 else 0.0

    # 2. List Most Common Failure Modes
    # Filter for failed tasks only
    failed_df = df[df['Success'] == 0]
    if not failed_df.empty:
        failure_counts = failed_df['Failure Reason'].value_counts()
        top_failures = failure_counts.to_string()
    else:
        top_failures = "None (All tasks successful!)"

    # 3. Format the Report
    report = f"""
=================================================================
                OSWORLD EVALUATION SHORT REPORT
=================================================================

1. OVERALL SUCCESS RATE
-----------------------
* Total Tasks Evaluated: {total_tasks}
* Successful Tasks:      {success_count}
* Success Rate:          {success_rate:.1f}%
* Average Steps Taken:   {df['Steps'].mean():.1f}

2. COMMON FAILURE MODES
-----------------------
{top_failures}

3. DOMAIN BREAKDOWN
-------------------
{df.groupby('Domain')[['Success', 'Steps']].mean().to_string()}

=================================================================
"""
    return report


In [2]:

if __name__ == "__main__":


    # Auto-detect latest run if not provided
    target_dir = "/Users/mvaishak/Developer/AIAgents/myTestAgentV4/results/run_20251121_154544"
    if not target_dir:
        all_runs = glob.glob("results/run_*")
        if all_runs:
            target_dir = max(all_runs, key=os.path.getctime)
            print(f"Auto-detected latest run: {target_dir}")
        else:
            print("No run directories found in 'results/'.")
            exit(1)

    if os.path.exists(target_dir):
        df = load_run_data(target_dir)
        print(generate_summary_text(df))
    else:
        print(f"Directory not found: {target_dir}")

Scanning /Users/mvaishak/Developer/AIAgents/myTestAgentV4/results/run_20251121_154544...
Found 10 trajectory files.

                OSWORLD EVALUATION SHORT REPORT

1. OVERALL SUCCESS RATE
-----------------------
* Total Tasks Evaluated: 10
* Successful Tasks:      0
* Success Rate:          0.0%
* Average Steps Taken:   6.0

2. COMMON FAILURE MODES
-----------------------
Failure Reason
Wrong Result (Agent claimed success but eval failed)    4
Agent Gave Up (Self-Reported)                           3
Timeout (Max Steps Reached)                             3

3. DOMAIN BREAKDOWN
-------------------
                    Success  Steps
Domain                            
chrome                  0.0    5.5
libreoffice_calc        0.0    5.5
libreoffice_writer      0.0    2.5
os                      0.0    7.0
vscode                  0.0    9.5


